<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->
# Mission Sequence
---
Last revised by Z. Ellis on 2026 APR 6

## Objectives
This tutorial will demonstrate 

## Imports and Set Up

Here we'll import the necessary libraries and load in the tutorials data folder. Then we define units and frames and load a metakernel to use for time conversions and provide attitude information.

In [1]:
import scarabaeus as scb
import supplementary as supp

import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = supp.load_data()

## units, frames, kernels
km, hr = scb.Units.get_units(['km', 'hr'])
J2000 = scb.Frame('J2000')

scb.SpiceManager.clear_kernels()    # ensure clean kernel pool
scb.SpiceManager.load_kernel_from_mkfile(data.OREx_mk.path)
scb.SpiceManager.print_kernels()    # verify C-Kernels for the S/C (sc) and solar arrays (sa)

SCB tutorial data up to date.
                                 Kernels Loaded:
Source:   /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/tutorials/supplementary/data/kernels/scenario/orex_mk.tm   (META)
Source:   data/kernels/scenario/orx_sc_rel_210816_210822_v02.bc   (CK)
Source:   data/kernels/scenario/orx_sa_rel_210816_210822_v02.bc   (CK)
Source:   data/kernels/scenario/ORX_SCLKSCET_00075.tsc   (TEXT)
Source:   data/kernels/locked/lsk/naif0012.tls   (TEXT)
Source:   data/kernels/scenario/orx_v14.tf   (TEXT)


In [2]:
# set up
km, sec, kg = scb.Units.get_units(['km', 'sec', 'kg'])
J2000 = scb.Frame('J2000')

t0 = scb.SpiceManager.cal2et('2026 JAN 01 00:00:00.000')
tf = scb.SpiceManager.cal2et('2026 JAN 02 00:00:00.000')
epochs = scb.EpochArray(np.arange(t0, tf, 500), 'TDB')

# bodies
jupiter = scb.CelestialBody.from_constants('JUPITER')
sc = scb.Spacecraft('TEST SC', 10000, tot_mass = scb.ArrayWUnits(1e3, kg))

pos0 = scb.ArrayWFrame(scb.ArrayWUnits([25e6, 0, 0], km), J2000)
vel0 = scb.ArrayWFrame(scb.ArrayWUnits([3, 0, 0], km/sec), J2000)

state_vector = scb.StateArray(epoch  = scb.EpochArray(scb.ArrayWUnits(t0, sec), 'TDB'),
                              origin = jupiter,
                              state  = scb.StateDefinition()
                                          .position(sc, pos0)
                                          .velocity(sc, vel0))

# propagator
fm = scb.ForceModelTranslation(sc)
prop = scb.Propagator(primary_body  = sc, 
                      state_vector  = state_vector, 
                      tspan         = epochs, 
                      force_models  = fm, 
                      propagate_STM = False)

# mission sequence
sequence = scb.MissionSequence('TEST_SEQUENCE')
sequence.addLeg('LEG1', scb.ArrayWUnits(86400, sec), state_vector, prop, scb.ArrayWUnits(10, sec))

mission = sequence.propagate()

# NOTE: cant specify file location so this will fail when it tries to write the file
orbiter_traj = scb.Trajectory('TEST_TRAJ', state_array = mission.propagated_state_array)

TypeError: Unsupported time input type: <class 'scarabaeus.units.ArrayWUnits.ArrayWUnits'>.